# Phase 2 · Data and baselines (E0)

Builds the question/secret datasets, then measures how far the model gets **without any training**:

| Method | What it is |
|---|---|
| `no_secret` | The model's normal answer, scored against the secret. The chance floor. |
| `prompted` | The rule explained in the prompt, plus one worked example. |
| `constrained` | Same prompt, but each sentence is forced to start with the right letter. The channel's upper bound. |

It ends with the phase 2 exit check: can constrained decoding produce **verified** training examples for phase 3?

**Runtime:** *Runtime → Change runtime type → T4 GPU*. Everything is saved to Google Drive as it goes. If Colab disconnects, reconnect and run all cells again; finished work is skipped.

Rough timing on a free T4 with the default settings (Qwen3-1.7B, 100 examples per split): 20–40 minutes. The 4B model takes roughly 2–3× longer.

In [ ]:
import os, sys

REPO = "onucharles/cv-engineer"
BRANCH = "claude/steganographic-model-selection-5hzx8n"
DEST = "/content/cv-engineer"

try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:  # not on Colab, or no secret set: fine for a public repo
    token = None

url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
if os.path.exists(DEST):
    !git -C {DEST} pull -q
else:
    !git clone -q --branch {BRANCH} {url} {DEST}

%cd {DEST}
!pip install -q -e ".[dev,gen]"
# Colab's preinstalled torchao is too old for current PEFT and breaks importing it.
!pip uninstall -y -q torchao
sys.path.insert(0, DEST)

## Settings

Start with **Qwen3-1.7B** to check everything works, then rerun with the main model, `Qwen/Qwen3-4B-Instruct-2507`. Each model gets its own run folder, and the dataset is shared.

In [ ]:
MODEL = "Qwen/Qwen3-1.7B"          # then: "Qwen/Qwen3-4B-Instruct-2507"
CHANNEL = "acrostic"               # "parity" skips constrained decoding
N_PER_SPLIT = 100                  # eval examples per split (4 splits)
BATCH_SIZE = 16                    # lower to 8 if you run out of GPU memory

In [ ]:
import torch
from stego.colab import run_dir

assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → T4 GPU"
print(torch.cuda.get_device_name(0))

DATA_DIR = run_dir("00_data")
RUN_DIR = run_dir(f"02_baselines-{CHANNEL}-{MODEL.split('/')[-1]}")
print("data:", DATA_DIR, "\nrun: ", RUN_DIR)

## 1. Build the datasets

Downloads Dolly-15k, keeps questions that can be answered in plain prose, and pairs them with secret words. Held-out splits test unseen words, longer words, and a different kind of question.

In [ ]:
from stego.experiments import baselines as e0
from stego.io import read_jsonl

examples_path = e0.build_datasets(DATA_DIR, channel=CHANNEL, n_eval=200)
examples = read_jsonl(examples_path)
rows = e0.eval_rows(examples, N_PER_SPLIT)
print(len(examples), "examples;", len(rows), "used for the baselines")
for e in examples[:3]:
    print(e["split"], "|", e["secret"], "|", e["question"])

## 2. Load the model

In [ ]:
from stego.generation.runner import load_model
from stego.generation.constrained import AcrosticVocab

model, tokenizer = load_model(MODEL)
vocab = AcrosticVocab(tokenizer, model.config.vocab_size) if CHANNEL == "acrostic" else None
print(model.dtype, f"{torch.cuda.memory_allocated() / 1e9:.1f} GB on GPU")

## 3. Reference answers

Two answers per question, both with no secret:

- **greedy**: the model's single most likely answer. This is the `no_secret` baseline.
- **sampled** (temperature 1, no truncation): the model's *typical* answer. Fluency is measured against this. Greedy text is far more probable than any normal text, so measuring against it made every answer look disfluent.

In [ ]:
references = e0.generate_references(RUN_DIR, rows, model, tokenizer, batch_size=BATCH_SIZE)
fluency_refs = e0.generate_references(RUN_DIR, rows, model, tokenizer, sampled=True, batch_size=BATCH_SIZE)

## 4. Baselines

In [ ]:
e0.run_baselines(RUN_DIR, rows, model, tokenizer, vocab, batch_size=BATCH_SIZE)

## 5. Results

- **exact**: share of answers that decode to the whole secret
- **letter_acc**: partial credit (edit distance)
- **clean_format**: plain prose with no sentence under five words
- **fluency**: 1 means at least as natural as the model's typical answer; 0 means 1+ nats/token less likely
- **lp_gap**: the raw difference in nats/token behind fluency (0 = as natural, negative = less)
- **exact_and_clean**: both at once, the number that matters
- **hard_exact**: exact rate on secrets containing Q, X, Y or Z

In [ ]:
from IPython.display import Markdown, display
from stego.eval.scoring import markdown_table

table = e0.score_baselines(RUN_DIR, rows, references, model, tokenizer, fluency_refs=fluency_refs)
display(Markdown(markdown_table(table)))

In [ ]:
for method in ["prompted", "constrained"]:
    print(e0.show_examples(RUN_DIR / f"eval_answers_{method}_scored.jsonl", n=3))

## 6. Exit check: verified SFT demonstrations

Samples 4 constrained answers for 32 training prompts and keeps the most fluent one that decodes exactly, is clean prose and has fluency ≥ 0.5. Phase 3 runs this at full scale to build the SFT dataset.

In [ ]:
train = [e for e in examples if e["split"] == "train"]
demos = e0.sft_demo_smoke_test(RUN_DIR, train, model, tokenizer, vocab, n_prompts=32, samples=4, batch_size=BATCH_SIZE)
for d in demos[:3]:
    print(f"secret={d['secret']}  fluency={d['fluency']:.2f}  ({d['n_verified']}/{d['n_candidates']} verified)")
    print(d["question"]); print(d["answer"]); print()

## Phase 2 is done when

- the baseline table above is filled for the main model, and
- the exit check produces verified demonstrations for a reasonable share of prompts.

Copy `eval_baselines_summary_table.md` from the run folder in Drive (`MyDrive/stego-runs/...`) if you want to share the numbers. Phase 3 builds the full SFT dataset and trains.